# Notebook 1 — Data Preparation
 
 **Goal**: Load raw loan data, clean it, filter to (EE) customers, and save a clean parquet file for target construction.

## Import libraries 

In [1]:
%load_ext autoreload
%autoreload 2

import pandas as pd
import numpy as np
import sys, os

# Add project src/ to path
sys.path.insert(0, os.path.abspath('../src'))

# Now, even if you change config.py, the next cell run will pick it up
from utils.config import PATHS, MISSINGNESS_DROP_PCT,LEAKAGE_COLUMNS
from utils.data_cleaning import (
    load_raw_data,
    drop_empty_columns,
    drop_constant_columns,
    detect_date_columns,
    convert_date_columns,
    find_sentinel_columns,
    replace_sentinel_with_nan,
    missingness_summary,
    drop_high_missing,
    find_post_default_numeric,
    find_post_default_categorical,
    find_future_date_columns,
    # Data quality tests
    test_unique_ratio,
    test_near_zero_variance,
    test_temporal_leak,
)

# Load Dataset

In [2]:
df = load_raw_data(PATHS['raw_data'])
data_dictionary = pd.read_csv('../references/data_dictionary.csv')

Raw dataset: 332,711 rows × 112 columns


In [4]:
# Quick sanity check
print(f"Duplicate rows: {df.duplicated().sum()}")

Duplicate rows: 0


In [4]:
df['LoanDate'].max()

'2023-10-14'

In [5]:
df['LoanDate'].min()

'2009-02-28'

# General Processing

In [5]:
# Drop fully empty columns
df, empty_cols = drop_empty_columns(df)

Dropped 4 fully empty columns: ['DateOfBirth', 'County', 'City', 'EmploymentPosition']


In [6]:
# Detect and convert date columns
date_cols = detect_date_columns(df)
df = convert_date_columns(df, date_cols)

Detected 18 date columns: ['ReportAsOfEOD', 'ListedOnUTC', 'BiddingStartedOn', 'LoanApplicationStartedDate', 'LoanDate', 'ContractEndDate', 'FirstPaymentDate', 'MaturityDate_Original', 'MaturityDate_Last', 'LastPaymentOn', 'DebtOccuredOn', 'DebtOccuredOnForSecondary', 'DefaultDate', 'StageActiveSince', 'GracePeriodStart', 'GracePeriodEnd', 'NextPaymentDate', 'ReScheduledOn']
--- Before conversion ---
ReportAsOfEOD                 str
ListedOnUTC                   str
BiddingStartedOn              str
LoanApplicationStartedDate    str
LoanDate                      str
ContractEndDate               str
FirstPaymentDate              str
MaturityDate_Original         str
MaturityDate_Last             str
LastPaymentOn                 str
DebtOccuredOn                 str
DebtOccuredOnForSecondary     str
DefaultDate                   str
StageActiveSince              str
GracePeriodStart              str
GracePeriodEnd                str
NextPaymentDate               str
ReScheduledOn     

# Filter to Estonian Customers

In [7]:
df_ee = df[df['Country'] == 'EE'].copy()
print(f"\nEstonian portfolio: {len(df_ee):,} loans")


Estonian portfolio: 162,573 loans


# Missingness Anlaysis

We check how many features fall into each missingness bucket.

Features with >90% missing are dropped — they add noise, not signal.

In [8]:
summary, missing_pct = missingness_summary(df_ee, title="Estonian Portfolio")


### Estonian Portfolio ###
+---------------------+----------------------+
| Missingness Range   |   Number of Features |
|---------------------+----------------------|
| 0%                  |                   43 |
| 0.1-10%             |                   24 |
| 10-20%              |                    1 |
| 20-30%              |                    1 |
| 30-40%              |                    3 |
| 40-50%              |                    1 |
| 50-60%              |                    3 |
| 60-70%              |                   21 |
| 70-80%              |                    2 |
| 80-90%              |                    2 |
| 90-100%             |                    7 |
+---------------------+----------------------+

Features with >90% missingness : ['Rating_V2', 'Rating_V1', 'EL_V1', 'Rating_V0', 'EL_V0', 'CreditScoreFiAsiakasTietoRiskGrade', 'CreditScoreEsEquifaxRisk']


# Sentinel Value Handling

The data dictionary notes that `-1` was used as a placeholder for missing values in several columns. We replace these with proper NaN so they're handled correctly downstream. 

`FreeCash` is excluded because negative free cash is a real value.

In [9]:
sentinel_cols = find_sentinel_columns(df_ee, sentinel=-1)
print(sentinel_cols)

Found 7 columns with sentinel value -1:
OccupationArea       140174
MaritalStatus        140171
UseOfLoan            140171
EmploymentStatus     140171
Education               574
HomeOwnershipType         2
FreeCash                  1
['UseOfLoan', 'Education', 'MaritalStatus', 'EmploymentStatus', 'OccupationArea', 'HomeOwnershipType', 'FreeCash']


A lot of categorical columns also have empty values

In [10]:
df_ee = replace_sentinel_with_nan(df_ee, sentinel_cols, sentinel=-1, exclude=['FreeCash'])

Replaced -1 → NaN in 6 columns (excluded: ['FreeCash'])


# Leakage Detection

In [11]:
# Numeric columns empty for healthy loans
proxy_numeric = find_post_default_numeric(df_ee)

Post-default numeric proxies (empty when healthy): ['PlannedPrincipalPostDefault', 'PlannedInterestPostDefault', 'EAD1', 'EAD2', 'PrincipalRecovery', 'InterestRecovery']


In [12]:
# Categorical columns empty for healthy loans
proxy_cat = find_post_default_categorical(df_ee)

Post-default categorical proxies: []


In [13]:
# Date columns occurring after LoanDate (post-origination events)
future_dates = find_future_date_columns(df_ee)

Future date columns (post-origination events): ['ReportAsOfEOD', 'ListedOnUTC', 'BiddingStartedOn', 'LoanApplicationStartedDate', 'ContractEndDate', 'FirstPaymentDate', 'MaturityDate_Original', 'MaturityDate_Last', 'LastPaymentOn', 'DebtOccuredOn', 'DebtOccuredOnForSecondary', 'DefaultDate', 'StageActiveSince', 'GracePeriodStart', 'GracePeriodEnd', 'NextPaymentDate', 'ReScheduledOn']


In [14]:
# Drop future date columns except DefaultDate (needed for target)
future_dates_drop = [c for c in future_dates if c not in ['DefaultDate']]

In [15]:
df_ee.drop(columns=future_dates_drop, inplace=True)
print(f"Dropped {len(future_dates_drop)} post-origination date columns which are :{future_dates_drop}")

Dropped 16 post-origination date columns which are :['ReportAsOfEOD', 'ListedOnUTC', 'BiddingStartedOn', 'LoanApplicationStartedDate', 'ContractEndDate', 'FirstPaymentDate', 'MaturityDate_Original', 'MaturityDate_Last', 'LastPaymentOn', 'DebtOccuredOn', 'DebtOccuredOnForSecondary', 'StageActiveSince', 'GracePeriodStart', 'GracePeriodEnd', 'NextPaymentDate', 'ReScheduledOn']


# Zero-Variance Columns

After filtering to EE, some columns become constant (e.g., Country = 'EE').

In [16]:
df_ee, constant_cols = drop_constant_columns(df_ee, protect=['DefaultDate'])

Zero-variance columns (3 found, 3 dropped): ['Country', 'CreditScoreEsMicroL', 'CreditScoreEsEquifaxRisk']


In [17]:
# %%
print("=" * 70)
print("DATA QUALITY TESTS")
print("=" * 70)

# Test 1: Identifier detection
_ = test_unique_ratio(df_ee, threshold=0.90)

# Test 2: Near-zero variance
_ = test_near_zero_variance(df_ee, dominant_pct_threshold=90)

# Test 3: Temporal leak
_ = test_temporal_leak(df_ee)

DATA QUALITY TESTS

Columns with unique_ratio > 0.9 (potential identifiers):
    column  n_unique  unique_ratio
    LoanId    162573           1.0
LoanNumber    162573           1.0

Columns with dominant value >90% (near-zero variance):
                                column  n_unique  dominant_value_pct
    CreditScoreFiAsiakasTietoRiskGrade         3               100.0
                    IncomeFromLeavePay       325                99.6
                IncomeFromChildSupport       135                99.4
               IncomeFromSocialWelfare       163                99.3
                     IncomeFromPension       355                98.6
                             Rating_V0         8                98.2
                                 EL_V0       826                98.2
                           IncomeOther       488                97.3
             IncomeFromFamilyAllowance       239                97.3
                  RefinanceLiabilities        21                96.2
   

In [18]:
# i saved all the columns to be dropped in config.py 
# Dopping them all here , except DefaultDate and days_to_default which I need in next steps
to_drop = list(set(LEAKAGE_COLUMNS) - set(['DefaultDate', 'days_to_default']))

df_ee.drop(columns=to_drop, inplace=True, errors='ignore')

In [19]:
print(f'We have now : {len(df_ee.columns.to_list())} variables remaining')

We have now : 43 variables remaining


Checking if LoanDuration was stamped at origination

In [20]:
#  Checking if LoanDuration is a leakage or not

df['loan_term_original'] = (
    (pd.to_datetime(df['MaturityDate_Original']) - pd.to_datetime(df['LoanDate']))
    .dt.days / 30.44
).round()

df['loan_term_current'] = (
    (pd.to_datetime(df['MaturityDate_Last']) - pd.to_datetime(df['LoanDate']))
    .dt.days / 30.44
).round()

df['matches_original'] = np.isclose(df['LoanDuration'], df['loan_term_original'], atol=1)
df['matches_current']  = np.isclose(df['LoanDuration'], df['loan_term_current'],  atol=1)

print(df[['LoanDuration', 'loan_term_original', 'loan_term_current', 
          'matches_original', 'matches_current', 'Restructured']].head(20))


    LoanDuration  loan_term_original  loan_term_current  matches_original  \
0             60                60.0               38.0              True   
1             60                59.0              121.0              True   
2             60                61.0               51.0              True   
3            108               107.0              107.0              True   
4             60                59.0               59.0              True   
5             60                60.0              129.0              True   
6             60                60.0               60.0              True   
7             60                61.0               61.0              True   
8            119               118.0              118.0              True   
9             60                60.0               60.0              True   
10            60                60.0               60.0              True   
11            60                60.0              123.0              True   

# Export Cleaned Data

In [21]:

df_ee.to_parquet(PATHS['cleaned'], index=False)
print(f"\n✓ Saved cleaned EE data → {PATHS['cleaned']}")
print(f"  Shape: {df_ee.shape[0]:,} rows × {df_ee.shape[1]} columns")


✓ Saved cleaned EE data → /home/me/Documents/Coding/Projects/pd_model/data/02_processed/01_estonia_cleaned.parquet
  Shape: 162,573 rows × 43 columns
